# Notebook 01 — Data Preparation 

Pipeline order (strict — no computation before missingness is resolved):

1. Load raw data and rename truncated oTree column names
2. Map treatment string labels to integer codes
3. Apply pre-registered exclusions (incomplete sessions, missing ideology)
4. Define analytic sample — listwise deletion on all analysis variables
5. Compute composites and z-standardise on the fixed sample
   - Recode political and electoral variables
7. Export missingness audit and clean CSVs

**Outputs**: `data/mturk_clean.csv` · `data/prolific_clean.csv`

> `voted_trump_2020` non-voter `NaN` values are assigned *by design* in Cell 6 and are not part of the analytic sample boundary. The R2 robustness run handles them with a targeted drop inside Notebook 05.

In [1]:
# Cell 1: Imports
import pandas as pd
import numpy as np
import pingouin as pg
import os, warnings
warnings.filterwarnings('ignore')

os.makedirs('data', exist_ok=True)
os.makedirs('outputs/tables', exist_ok=True)
SEED = 42
np.random.seed(SEED)


In [2]:
# Cell 2: Scale constants and column-name registry

FIVE_PT_MAX = 5   # five_point_scale(): 1=Strongly disagree, 5=Strongly agree
LIBERAL_MAX = 7   # liberal: 1=Extremely liberal, 7=Extremely conservative

# Rename truncated oTree columns to full descriptive names
RENAME = {
    'cpr1groupgrp_treat_num': 'treatment',
    'environment_responsib':  'environment_responsibility',
    'environment_natural_p':  'environment_natural_process',
    # Treatment dummy columns
    'cpr1groupgrp_treat_num_1': 'treat_IT_GI',
    'cpr1groupgrp_treat_num_2': 'treat_IT',
    'cpr1groupgrp_treat_num_3': 'treat_GI',
    'cpr1groupgrp_treat_num_4': 'treat_NIT_NGI',
    # Extraction dummy columns
    'extraction_1':             'extract_no_harvest',
    'extraction_2':             'extract_green_choice',
    'extraction_3':             'extract_unsustainable_choice',
}

# Mapped to canonical integer coding:
#   1 = IT&GI  |  2 = IT only  |  3 = GI only  |  4 = NIT&NGI (pure control)
TREATMENT_STR_MAP = {
    'IT&GI':   1, 'IT': 2, 'GI': 3, 'NIT&NGI': 4,
    1: 1, 2: 2, 3: 3, 4: 4,          # already-integer pass-through
    '1': 1, '2': 2, '3': 3, '4': 4,  # float-as-string pass-through
}

# Legacy concern items — six-item scale; items 5–6 negatively worded
LEGACY_ITEMS   = [f'legacy_concern{i}' for i in range(1, 7)]
LEGACY_REVERSE = ['legacy_concern5', 'legacy_concern6']

# Environmental concern: 5 subscales × 6 items; reverse items per instrument
ALL_ENV_SUBSCALES = [
    ([f'environment_mov{i}'    for i in range(1,7)],
     ['environment_mov2','environment_mov3','environment_mov5'], 'env_mov'),
    ([f'environment_threat{i}' for i in range(1,7)],
     ['environment_threat4','environment_threat5','environment_threat6'], 'env_threat'),
    ([f'environment_beha{i}'   for i in range(1,7)],
     ['environment_beha1','environment_beha2','environment_beha5'], 'env_beha'),
    ([f'environment_uti{i}'    for i in range(1,7)],
     ['environment_uti1','environment_uti5','environment_uti6'], 'env_uti'),
    ([f'environment_growth{i}' for i in range(1,7)],
     ['environment_growth2','environment_growth4','environment_growth6'], 'env_growth'),
]
ALL_ENV_ITEMS = [item for items,_,_ in ALL_ENV_SUBSCALES for item in items]

In [3]:
# Cell 3: Load raw data, rename columns, map treatment to integer codes
df_mt = pd.read_csv('data/Data_decisions_MA_Mturk.csv').rename(columns=RENAME)
df_pr = pd.read_csv('data/Data_decisions_MA_Prolific.csv').rename(columns=RENAME)

# Record raw counts for missingness audit
N_RAW_MT_ROWS = len(df_mt)
N_RAW_PR_ROWS = len(df_pr)
N_RAW_MT_PT   = df_mt['participantcode'].nunique()
N_RAW_PR_PT   = df_pr['participantcode'].nunique()

# Map treatment string labels → integers.
for df in [df_mt, df_pr]:
    df['treatment'] = df['treatment'].map(TREATMENT_STR_MAP)
    n_unmapped = df['treatment'].isna().sum()
    if n_unmapped > 0:
        print(f'  WARNING: {n_unmapped} rows have unrecognised treatment values → NaN')

# Edu_* binary dummies are retained in the raw data but were found
# to be inconsistent with the 'education' column and are therefore excluded
# from all analyses. The ordinal 'education' column
# (values 1–7) is used throughout.
edu_cols = [f'Edu_{i}' for i in range(1, 8)]
for df, label in [(df_mt, 'MTurk'), (df_pr, 'Prolific')]:
    df.drop(columns=[c for c in edu_cols if c in df.columns], inplace=True)

df_mt['treatment'] = df_mt['treatment'].astype(int)
df_pr['treatment'] = df_pr['treatment'].astype(int)

print(f'Raw MTurk:    {N_RAW_MT_ROWS:,} rows | {N_RAW_MT_PT:,} participants')
print(f'Raw Prolific: {N_RAW_PR_ROWS:,} rows | {N_RAW_PR_PT:,} participants')
print('Treatment value counts (MTurk):')
print(df_mt['treatment'].value_counts().sort_index())
print('Treatment value counts (Prolific):')
print(df_pr['treatment'].value_counts().sort_index())
print("Edu_* dummy columns dropped from both datasets.")

Raw MTurk:    11,389 rows | 1,167 participants
Raw Prolific: 10,751 rows | 1,093 participants
Treatment value counts (MTurk):
treatment
1    2962
2    2605
3    2791
4    3031
Name: count, dtype: int64
Treatment value counts (Prolific):
treatment
1    2762
2    2389
3    3110
4    2490
Name: count, dtype: int64
Edu_* dummy columns dropped from both datasets.


## Step 3 — Pre-Registered Exclusions

1. **Incomplete sessions**: participants without all 10 decision rounds excluded.
2. **Missing primary ideology measure** (`liberal`): participants missing `liberal` cannot be placed on the ideology continuum and are excluded from all models.

In [4]:
# Cell 4: Pre-registered exclusions
def apply_prereg_exclusions(df, label):
    n0 = df['participantcode'].nunique()
    complete_ids = (
        df.groupby('participantcode')['round']
        .count()
        .pipe(lambda s: s[s == 10].index)
    )
    df = df[df['participantcode'].isin(complete_ids)]
    df = df[df['liberal'].notna()]
    n1 = df['participantcode'].nunique()
    print(f'  {label}: {n0:,} → {n1:,} participants ({n0-n1:,} excluded by pre-reg criteria)')
    return df

df_mt = apply_prereg_exclusions(df_mt, 'MTurk')
df_pr = apply_prereg_exclusions(df_pr, 'Prolific')


  MTurk: 1,167 → 1,033 participants (134 excluded by pre-reg criteria)
  Prolific: 1,093 → 1,017 participants (76 excluded by pre-reg criteria)


## Step 4 — Analytic Sample Boundary (Listwise Deletion)

All rows missing on **any variable used across the full analysis** are dropped here, before any composite or z-score is computed. This single operation fixes the analytic sample so that every z-score, Cronbach's α, and model is estimated on the identical N.

**Exception**: `voted_trump_2020` is excluded from this drop list because its `NaN` values represent non-voters assigned by design in Cell 6, not incidental missingness. Robustness Run R2 in Notebook 05 handles them with a targeted drop.

In [5]:
# Cell 5: Define analytic sample — listwise deletion BEFORE any computation
ALWAYS_REQUIRED_MT = [
    'participantcode', 'sessioncode', 'round',
    'extract_green_choice', 'treatment',
    'liberal', 'political_orientation',
    'age', 'gender', 'education', 'ethnic_min',
    'trust_scientists',
]

PROLIFIC_EXTRA = [
    'election_2020', 'election_2024',
    'risk_attitude', 'short_long_reward',
] + LEGACY_ITEMS + ALL_ENV_ITEMS

# Record post-pre-reg counts for audit
N_PREREG_MT_ROWS = len(df_mt)
N_PREREG_PR_ROWS = len(df_pr)
N_PREREG_MT_PT   = df_mt['participantcode'].nunique()
N_PREREG_PR_PT   = df_pr['participantcode'].nunique()

df_mt = df_mt.dropna(subset=ALWAYS_REQUIRED_MT)
df_pr = df_pr.dropna(subset=ALWAYS_REQUIRED_MT + PROLIFIC_EXTRA)

print('Analytic sample after listwise deletion:')
print(f'  MTurk:    {len(df_mt):,} rows | {df_mt["participantcode"].nunique():,} participants')
print(f'  Prolific: {len(df_pr):,} rows | {df_pr["participantcode"].nunique():,} participants')
print()
print('Rows dropped by listwise deletion:')
print(f'  MTurk:    {N_PREREG_MT_ROWS - len(df_mt):,} '
      f'({(N_PREREG_MT_ROWS-len(df_mt))/N_PREREG_MT_ROWS*100:.1f}%)')
print(f'  Prolific: {N_PREREG_PR_ROWS - len(df_pr):,} '
      f'({(N_PREREG_PR_ROWS-len(df_pr))/N_PREREG_PR_ROWS*100:.1f}%)')


Analytic sample after listwise deletion:
  MTurk:    10,220 rows | 1,022 participants
  Prolific: 10,170 rows | 1,017 participants

Rows dropped by listwise deletion:
  MTurk:    110 (1.1%)
  Prolific: 0 (0.0%)


## Step 5a — Political Orientation Variables

All z-standardisations below are computed on the fixed analytic sample.

- **`liberal_z`**: z-standardised ideology (1=Liberal → 7=Conservative). Primary moderator.
- **`party_id_z`**: z-standardised party ID (1–8). Code 8 (Other party) retained — excluding it would introduce selection bias. Robustness Run R1 only.
- **`voted_trump_2020`** / **`voted_rep_2024`** (Prolific only): binary vote-choice proxies. `voted_trump_2020` assigns `NaN` to non-voters by design.

In [6]:
# Cell 6: Political variable recoding — on fixed analytic sample
def recode_political(df, is_prolific=False):
    df = df.copy()
    df['liberal_z'] = (df['liberal'] - df['liberal'].mean()) / df['liberal'].std(ddof=1)
    df['party_id_z'] = (
        (df['political_orientation'] - df['political_orientation'].mean())
        / df['political_orientation'].std(ddof=1)
    )
    if is_prolific:
        # R2: non-voters assigned NaN by design — not analytic missingness
        df['voted_trump_2020'] = df['election_2020'].map({1: 0, 2: 1, 3: np.nan})
        # R3: all six 2024 candidates mapped Dem=0 / Rep=1; no exclusions
        df['voted_rep_2024']   = df['election_2024'].map({1:0, 5:0, 6:0, 2:1, 3:1, 4:1})
    return df

df_mt = recode_political(df_mt, is_prolific=False)
df_pr = recode_political(df_pr, is_prolific=True)

for name, df in [('MTurk', df_mt), ('Prolific', df_pr)]:
    print(f'  {name} — liberal_z mean: {df["liberal_z"].mean():.8f}, '
          f'party_id_z mean: {df["party_id_z"].mean():.8f}')


  MTurk — liberal_z mean: -0.00000000, party_id_z mean: -0.00000000
  Prolific — liberal_z mean: 0.00000000, party_id_z mean: -0.00000000


## Step 5b — Legacy Concern Scale (Prolific only)

Six items from McAdams & de St. Aubin (1992) on `five_point_scale()` (1–5). Items 5–6 negatively worded; reversed = (5+1) − original. Cronbach's α at participant level. `legacy_z` z-standardised on the analytic sample.

In [7]:
# Cell 7: Legacy concern — Prolific only, on fixed analytic sample
def build_legacy(df, label):
    df = df.copy()
    for item in LEGACY_REVERSE:
        df[item] = (FIVE_PT_MAX + 1) - df[item]
    p_level = df.groupby('participantcode')[LEGACY_ITEMS].mean()
    alpha   = pg.cronbach_alpha(data=p_level[LEGACY_ITEMS])[0]
    print(f'  {label} legacy concern α = {alpha:.3f} (N={len(p_level):,} participants)')
    df['legacy_raw'] = df[LEGACY_ITEMS].mean(axis=1)
    df['legacy_z']   = (df['legacy_raw'] - df['legacy_raw'].mean()) / df['legacy_raw'].std(ddof=1)
    return df

df_pr       = build_legacy(df_pr, 'Prolific')
df_mt['legacy_z'] = np.nan   # schema placeholder — not collected in MTurk

print(f'  Prolific legacy_z: mean={df_pr["legacy_z"].mean():.8f}')


  Prolific legacy concern α = 0.856 (N=1,017 participants)
  Prolific legacy_z: mean=-0.00000000


## Step 5c — Environmental Concern Scale (Prolific only)

Five subscales × 6 items on `five_point_scale()` (1–5). Negatively worded items reverse-scored as (5+1) − original. `env_concern_z` is z-standardised on the analytic sample.

`environment_natural_process` code 6 (climate denial) is set to `NaN` — it cannot be placed on the natural-process/human-activity continuum.

In [8]:
# Cell 8: Environmental concern — Prolific only, on fixed analytic sample
def build_env_concern(df, label):
    df = df.copy()
    for items, rev_items, subscale_name in ALL_ENV_SUBSCALES:
        for item in rev_items:
            df[item] = (FIVE_PT_MAX + 1) - df[item]
        df[subscale_name] = df[items].mean(axis=1)
    p_level = df.groupby('participantcode')[ALL_ENV_ITEMS].mean()
    alpha   = pg.cronbach_alpha(data=p_level[ALL_ENV_ITEMS])[0]
    print(f'  {label} env concern α (30 items) = {alpha:.3f} (N={len(p_level):,} participants)')
    df['env_concern_raw'] = df[ALL_ENV_ITEMS].mean(axis=1)
    df['env_concern_z']   = (
        (df['env_concern_raw'] - df['env_concern_raw'].mean())
        / df['env_concern_raw'].std(ddof=1)
    )
    df['env_natural_process_ctrl'] = df['environment_natural_process'].replace(6, np.nan)
    return df

df_pr = build_env_concern(df_pr, 'Prolific')
df_mt['env_concern_z']           = np.nan
df_mt['env_natural_process_ctrl'] = df_mt['environment_natural_process'].replace(6, np.nan)

print(f'  Prolific env_concern_z: mean={df_pr["env_concern_z"].mean():.8f}')


  Prolific env concern α (30 items) = 0.927 (N=1,017 participants)
  Prolific env_concern_z: mean=0.00000000


In [9]:
# Cell 9: Additional continuous controls — z-standardised on analytic sample
# 11-choice variables stored as 1–11 (oTree 1-indexed).
# The constant offset of 1 is absorbed by z-standardisation.
def z_standardise(df, cols):
    df = df.copy()
    for col in cols:
        if col in df.columns:
            df[f'{col}_z'] = (df[col] - df[col].mean()) / df[col].std(ddof=1)
    return df

df_mt = z_standardise(df_mt, ['trust_scientists'])
df_pr = z_standardise(df_pr, ['trust_scientists', 'risk_attitude', 'short_long_reward'])

print('Z-score mean checks on analytic sample (all should be ≈ 0):')
for name, df, cols in [
    ('MTurk',    df_mt, ['trust_scientists_z']),
    ('Prolific', df_pr, ['trust_scientists_z','risk_attitude_z','short_long_reward_z']),
]:
    print(f'  {name}:', {c: round(df[c].mean(), 8) for c in cols})


Z-score mean checks on analytic sample (all should be ≈ 0):
  MTurk: {'trust_scientists_z': np.float64(-0.0)}
  Prolific: {'trust_scientists_z': np.float64(0.0), 'risk_attitude_z': np.float64(-0.0), 'short_long_reward_z': np.float64(-0.0)}


## Step 6 — Missingness Audit

Stages: (1) raw export, (2) after pre-registered exclusions, (3) after listwise deletion (analytic sample).

In [10]:
# Cell 10: Missingness audit — exported to outputs/tables/missingness_audit.csv
audit_rows = []
for study, raw_r, raw_p, prereg_r, prereg_p, final_df in [
    ('MTurk',    N_RAW_MT_ROWS, N_RAW_MT_PT, N_PREREG_MT_ROWS, N_PREREG_MT_PT, df_mt),
    ('Prolific', N_RAW_PR_ROWS, N_RAW_PR_PT, N_PREREG_PR_ROWS, N_PREREG_PR_PT, df_pr),
]:
    final_r = len(final_df)
    final_p = final_df['participantcode'].nunique()
    audit_rows.append({
        'Study':                      study,
        'Raw rows':                   raw_r,
        'Raw participants':           raw_p,
        'After pre-reg excl (rows)':  prereg_r,
        'After pre-reg excl (pts)':   prereg_p,
        'Analytic sample (rows)':     final_r,
        'Analytic sample (pts)':      final_p,
        'Dropped — pre-reg (rows)':   raw_r - prereg_r,
        'Dropped — listwise (rows)':  prereg_r - final_r,
        'Total dropped (%)':          f'{(raw_r - final_r)/raw_r*100:.1f}%',
    })

audit = pd.DataFrame(audit_rows).set_index('Study')
audit.T.to_csv('outputs/tables/missingness_audit.csv')
print('Missingness Audit (Table A1):')
print(audit.T.to_string())


Missingness Audit (Table A1):
Study                      MTurk Prolific
Raw rows                   11389    10751
Raw participants            1167     1093
After pre-reg excl (rows)  10330    10170
After pre-reg excl (pts)    1033     1017
Analytic sample (rows)     10220    10170
Analytic sample (pts)       1022     1017
Dropped — pre-reg (rows)    1059      581
Dropped — listwise (rows)    110        0
Total dropped (%)          10.3%     5.4%


In [11]:
# Cell 11: Export analytic datasets
# Downstream notebooks load these files. No further row-dropping is required
# except the targeted voted_trump_2020 drop in Notebook 05 (R2 only).
df_mt.to_csv('data/mturk_clean.csv',    index=False)
df_pr.to_csv('data/prolific_clean.csv', index=False)

print('Analytic datasets exported:')
print(f'  data/mturk_clean.csv    — {len(df_mt):,} rows, '
      f'{df_mt["participantcode"].nunique():,} participants')
print(f'  data/prolific_clean.csv — {len(df_pr):,} rows, '
      f'{df_pr["participantcode"].nunique():,} participants')

# Final sanity check: all z-scored composites must have mean ≈ 0 on exported data
z_mt = ['liberal_z', 'party_id_z', 'trust_scientists_z']
z_pr = z_mt + ['env_concern_z', 'legacy_z', 'risk_attitude_z', 'short_long_reward_z']
print()
print('Final z-score mean checks (all should be ≈ 0):')
print('  MTurk:   ', {c: round(df_mt[c].mean(), 8) for c in z_mt if c in df_mt})
print('  Prolific:', {c: round(df_pr[c].mean(), 8) for c in z_pr
                      if c in df_pr and df_pr[c].notna().any()})


Analytic datasets exported:
  data/mturk_clean.csv    — 10,220 rows, 1,022 participants
  data/prolific_clean.csv — 10,170 rows, 1,017 participants

Final z-score mean checks (all should be ≈ 0):
  MTurk:    {'liberal_z': np.float64(-0.0), 'party_id_z': np.float64(-0.0), 'trust_scientists_z': np.float64(-0.0)}
  Prolific: {'liberal_z': np.float64(0.0), 'party_id_z': np.float64(-0.0), 'trust_scientists_z': np.float64(0.0), 'env_concern_z': np.float64(0.0), 'legacy_z': np.float64(-0.0), 'risk_attitude_z': np.float64(-0.0), 'short_long_reward_z': np.float64(-0.0)}
